# DSC680 Week 11 Final Project: Procurement Analytics and Supplier Performance Optimization

**Student:** Kayla Greavu  
**Course:** DSC680-T301 Applied Data Science  

This JupyterLab notebook supports the final report and presentation. It loads the procurement dataset, cleans key fields, calculates procurement KPIs, creates visualizations, and builds a simple predictive model for delivery-delay risk.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, mean_absolute_error

plt.rcParams.update({'figure.dpi': 140, 'font.size': 10})
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
# Load data
# Place this notebook in the same folder as Dataset_Procurement(2).xlsx, or update the path below.
DATA_PATH = 'Dataset_Procurement(2).xlsx'

try:
    df = pd.read_excel(DATA_PATH, sheet_name='Data')
except FileNotFoundError:
    # Alternate path used when running from the ChatGPT workspace
    df = pd.read_excel('/mnt/data/Dataset_Procurement(2).xlsx', sheet_name='Data')

df.shape

In [ ]:
# Preview the data structure
df.head()

## 1. Data Cleaning and Preparation

The dataset contains procurement records with purchase order details, supplier information, pricing, delivery dates, invoice/payment status, contract type, and risk-related variables. The steps below convert dates, standardize Yes/No fields, and create flags used in analysis.

In [ ]:
# Convert date fields
for c in ['PO Date','Requested Delivery','Actual Delivery','Contract Start','Contract End']:
    df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')

# Standardize selected categorical fields
for c in ['On Time Delivery','Maverick Spend','Preferred Supplier','Single Source Flag']:
    df[c] = df[c].astype(str).str.strip().str.title()

# Create binary flags
df['on_time_flag'] = (df['On Time Delivery'] == 'Yes').astype(int)
df['late_flag'] = (df['Days Late'] > 0).astype(int)
df['maverick_flag'] = (df['Maverick Spend'] == 'Yes').astype(int)
df['preferred_flag'] = (df['Preferred Supplier'] == 'Yes').astype(int)
df['single_source_flag'] = (df['Single Source Flag'] == 'Yes').astype(int)

risk_order = {'Low': 1, 'Medium': 2, 'High': 3}
df['risk_score'] = df['Supplier Risk'].map(risk_order)

# Check missing values for key columns
key_cols = ['Supplier Name','Supplier Risk','Days Late','On Time Delivery','Savings Amount','Maverick Spend','Preferred Supplier','Supplier ESG Score']
df[key_cols].isna().sum()

## 2. Executive KPI Summary

In [ ]:
kpis = pd.Series({
    'Procurement records': len(df),
    'Unique suppliers': df['Supplier Name'].nunique(),
    'Total net spend': df['Line Net'].sum(),
    'Total spend including tax': df['Line Total Inc Tax'].sum(),
    'Total savings': df['Savings Amount'].sum(),
    'Average savings percent': df['Savings Pct'].mean(),
    'On-time delivery rate': df['on_time_flag'].mean(),
    'Average days late': df['Days Late'].mean(),
    'Maverick spend rate': df['maverick_flag'].mean(),
    'Maverick net spend': df.loc[df['maverick_flag']==1, 'Line Net'].sum()
})
kpis

## 3. Research Question 1: Which suppliers demonstrate the best overall performance?

The scorecard below uses delivery reliability, savings percentage, ESG score, and lower supplier risk. It is a business ranking tool, not a causal model.

In [ ]:
supplier = df.groupby('Supplier Name').agg(
    records=('PO Number','count'),
    on_time_rate=('on_time_flag','mean'),
    average_days_late=('Days Late','mean'),
    savings=('Savings Amount','sum'),
    average_savings_pct=('Savings Pct','mean'),
    average_esg_score=('Supplier ESG Score','mean'),
    average_risk_score=('risk_score','mean'),
    net_spend=('Line Net','sum')
).reset_index()

positive_savings_pct = supplier['average_savings_pct'].clip(lower=0)
supplier['performance_score'] = (
    supplier['on_time_rate'] * 40 +
    (positive_savings_pct / positive_savings_pct.max()) * 30 +
    (supplier['average_esg_score'] / 100) * 20 +
    ((3 - supplier['average_risk_score']) / 2) * 10
)

supplier_scorecard = supplier.sort_values('performance_score', ascending=False)
supplier_scorecard.head(10)

In [ ]:
top = supplier_scorecard.head(8).sort_values('performance_score')
plt.figure(figsize=(8,5))
plt.barh(top['Supplier Name'], top['performance_score'])
plt.xlabel('Composite performance score')
plt.title('Top Supplier Scorecard')
plt.tight_layout()
plt.show()

## 4. Research Question 2: What factors contribute most to late deliveries?

In [ ]:
risk_summary = df.groupby('Supplier Risk').agg(
    records=('PO Number','count'),
    on_time_rate=('on_time_flag','mean'),
    late_rate=('late_flag','mean'),
    average_days_late=('Days Late','mean'),
    average_savings_pct=('Savings Pct','mean'),
    net_spend=('Line Net','sum')
).reindex(['Low','Medium','High'])
risk_summary

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(risk_summary.index, risk_summary['average_days_late'])
plt.ylabel('Average days late')
plt.xlabel('Supplier risk level')
plt.title('Supplier Risk Level and Average Delivery Delay')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['Days Late','Lead Time Days','Savings Pct','Line Net','Supplier ESG Score','risk_score','preferred_flag','maverick_flag']
df[corr_cols].corr(numeric_only=True)['Days Late'].sort_values(ascending=False)

The strongest numeric relationship with delivery delay is lead time. Supplier risk, ESG score, savings percentage, preferred status, and maverick spend show weaker direct correlations in this dataset.

## 5. Research Question 3: How much money is tied to maverick spending?

In [ ]:
maverick_summary = df.groupby('Maverick Spend').agg(
    records=('PO Number','count'),
    share_of_records=('PO Number', lambda s: len(s)/len(df)),
    net_spend=('Line Net','sum'),
    average_line_net=('Line Net','mean'),
    average_days_late=('Days Late','mean'),
    average_savings_pct=('Savings Pct','mean')
)
maverick_summary

In [ ]:
plot = maverick_summary.loc[['No','Yes']]
plt.figure(figsize=(7,4))
plt.bar(plot.index, plot['average_days_late'])
plt.ylabel('Average days late')
plt.xlabel('Maverick spend')
plt.title('Maverick Spend and Delivery Delay')
plt.tight_layout()
plt.show()

## 6. Research Question 4: Are preferred suppliers providing measurable benefits?

In [ ]:
preferred_summary = df.groupby('Preferred Supplier').agg(
    records=('PO Number','count'),
    on_time_rate=('on_time_flag','mean'),
    average_days_late=('Days Late','mean'),
    average_savings_pct=('Savings Pct','mean'),
    total_savings=('Savings Amount','sum'),
    net_spend=('Line Net','sum')
)
preferred_summary

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(preferred_summary.index, preferred_summary['on_time_rate']*100)
plt.ylim(0,100)
plt.ylabel('On-time delivery rate (%)')
plt.xlabel('Preferred supplier status')
plt.title('Preferred Supplier Status vs. On-Time Delivery')
plt.tight_layout()
plt.show()

In this dataset, preferred suppliers do not clearly outperform non-preferred suppliers on delivery reliability. This suggests that the preferred supplier list should be reviewed and updated using actual performance KPIs.

## 7. Research Question 5: Can procurement data identify high-risk suppliers or trends?

In [ ]:
category = df.groupby('Category').agg(
    net_spend=('Line Net','sum'),
    savings=('Savings Amount','sum'),
    on_time_rate=('on_time_flag','mean'),
    average_days_late=('Days Late','mean'),
    records=('PO Number','count')
).sort_values('net_spend', ascending=False)
category

In [ ]:
cat = category.head(8).sort_values('net_spend')
plt.figure(figsize=(8,5))
plt.barh(cat.index, cat['net_spend']/1_000_000)
plt.xlabel('Net spend ($ millions)')
plt.title('Highest Procurement Spend by Category')
plt.tight_layout()
plt.show()

## 8. Predictive Modeling: Delivery Delay Risk

A logistic regression model estimates whether a procurement record is late. The purpose is to show how procurement data could support early risk flags, not to fully automate purchasing decisions.

In [ ]:
features_num = ['Lead Time Days', 'Savings Pct', 'Line Net', 'Supplier ESG Score', 'preferred_flag', 'maverick_flag', 'single_source_flag', 'risk_score']
features_cat = ['Supplier Risk', 'Contract Type', 'Local International', 'Category', 'Payment Status']
features = features_num + features_cat

X = df[features].copy()
y = df['late_flag']

preprocess = ColumnTransformer(transformers=[
    ('num', StandardScaler(), features_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), features_cat)
])
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:,1]
preds = (probs >= 0.5).astype(int)

pd.Series({
    'Accuracy': accuracy_score(y_test, preds),
    'ROC AUC': roc_auc_score(y_test, probs)
})

In [ ]:
# Linear regression for directional interpretation of standardized numeric predictors
Xn = df[features_num].fillna(0)
y_days = df['Days Late']
Xn_train, Xn_test, yr_train, yr_test = train_test_split(Xn, y_days, test_size=0.25, random_state=42)
lin = Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])
lin.fit(Xn_train, yr_train)
yr_pred = lin.predict(Xn_test)

coef = pd.DataFrame({
    'feature': features_num,
    'standardized_coefficient': lin.named_steps['model'].coef_
}).sort_values('standardized_coefficient', ascending=False)

print('R2:', round(r2_score(yr_test, yr_pred), 3))
print('MAE days:', round(mean_absolute_error(yr_test, yr_pred), 3))
coef

## 9. Export Summary Tables

In [ ]:
# Optional exports for reporting or presentation use
supplier_scorecard.to_csv('supplier_performance_scorecard.csv', index=False)
preferred_summary.to_csv('preferred_supplier_summary.csv')
risk_summary.to_csv('supplier_risk_summary.csv')
maverick_summary.to_csv('maverick_spend_summary.csv')
category.to_csv('category_summary.csv')
print('CSV summary files exported.')

## Final Recommendation Summary

1. Review and update preferred supplier status using actual KPIs, because preferred suppliers did not clearly outperform non-preferred suppliers in this dataset.
2. Monitor maverick spending, which represents 12.5% of records and over $98 million in net spend.
3. Build a supplier scorecard combining delivery reliability, savings, ESG score, and supplier risk.
4. Use lead time as an early warning indicator for delayed deliveries.
5. Extend the model in future work with more granular supplier and logistics variables.